### **I — Interface Segregation Principal(ISP)**
Definition: "Clients should not be forced to depend on methods they do not use"

In other words:
- Prefer many small, focused interfaces
- Avoid fat/god interfaces.

Why ISP matters in backend systems: <br>
In backend code, violations cause:
- Classes implementing methods they do not use
- Dummy implementations
- Runtime exceptions
- Broken LSP

**ISP** exists mainly to protect **LSP** !

**BAD DESIGN (Violates ISP)**

Example: Notification System

In [ ]:
class NotificationService:
    def send_email(self, msg):
        raise NotImplementedError
    
    def send_sms(self, msg):
        raise NotImplementedError

    def send_push(self, msg):
        raise NotImplementedError

Email Implementation

In [3]:
class EmailService(NotificationService):
    def send_email(self, msg):
        print(f"Send email with message: {msg}")
    
    def send_sms(self, msg):
        raise Exception("SMS not supported")
    
    def send_push(self, msg):
        raise Exception("Push notification is not supported")

In [6]:
email_service = EmailService()
email_service.send_email("Order confirmed!")
email_service.send_push("Order confirmed")

Send email with message: Order confirmed!


Exception: Push notification is not supported

**Why this is bad?**
- Client depends on unused methods
- Forces fake implementations
- Violates ISP
- Leads to LSP violations

**Correct Design (Follows ISP, LSP)**

Using Strategy Design Pattern

In [8]:
class EmailStrategy:
    def send(self, msg):
        raise NotImplementedError

class SMSStrategy:
    def send(self, msg):
        raise NotImplementedError

class PushStrategy:
    def send(self, msg):
        raise NotImplementedError

In [12]:
class EmailService(EmailStrategy):
    def send(self, msg):
        print(f"Send email with message: {msg}")

class SMSService(EmailStrategy):
    def send(self, msg):
        print(f"Send SMS with message: {msg}")

class PushService(EmailStrategy):
    def send(self, msg):
        print(f"Send Push Notification with message: {msg}")

In [13]:
def send_notification(sender, msg):
    sender.send(msg)

send_notification(PushService(), "Payment successful")

Send Push Notification with message: Payment successful


**Strategy + ISP = clean extension**

✔ No unused methods <br>
✔ Clean abstractions

**ISP + Adapter Pattern (3rd-party APIs/SDKs)**

ISP Violation

In [14]:
class PaymentGateway:
    def pay(self, amount):
        raise NotImplementedError

    def refund(self, amount):
        raise NotImplementedError

**Stripe Support both**

So, it follows Interface Segregation Principle

In [18]:
class StripeGateway(PaymentGateway):
    def pay(self, amount):
        print(f"Paid ${amount} via Stripe")
    
    def refund(self, amount):
        print(f"Refuned ${amount} successfully!")

In [19]:
stripe = StripeGateway()
stripe.pay(100)
stripe.refund(100)

Paid $100 via Stripe
Refuned $100 successfully!


Cash Payment does not support refund but pay

In [20]:
class CashPayment(PaymentGateway):
    def pay(self, amount):
        print(f"Paid ${amount} cash!")
    
    def refund(self, amount):
        raise Exception("Refund not available!")

In [22]:
cash = CashPayment()
cash.pay(100)
cash.refund(100)

Paid $100 cash!


Exception: Refund not available!

So, according to ISP, client should not be forced to implement unused method! So, it violates the principal.

**ISP FIX**

In [39]:
# Create a example Stripe SDKs
class StripePaymentGateway:
    def make_payment(self, amount):
        print(f"Payment ${100} transfered successfully!")
    
    def refund_request(self, amount):
        print(f"Your ${amount} refund request was successful. Your C/B: $150")

In [40]:
class Payable:
    def pay(self, amount):
        raise NotImplementedError

class Refundable:
    def refund(self, amount):
        raise NotImplementedError

In [41]:
class StripeAdapter(Payable, Refundable):
    def __init__(self):
        self.stripe = StripePaymentGateway()

    def pay(self, amount):
        self.stripe.make_payment(amount)
    
    def refund(self, amount):
        self.stripe.refund_request(amount)

class CashPayment(Payable):
    def pay(self, amount):
        print(f"Paid ${amount} Cash!")

In [42]:
stripe = StripeAdapter()
cash = CashPayment()

stripe.pay(100)
stripe.refund(100)

cash.pay(100)

Payment $100 transfered successfully!
Your $100 refund request was successful. Your C/B: $150
Paid $100 Cash!


✔ No fake methods <br>
✔ Adapter fits perfectly <br>

**Key ISP Rule (Memorize)**

If you are forced to implement a method with:
- pass
- raise NotImplementedError
- empty body

👉 You are violating ISP.

**ISP vs SRP**

| Principle | Focus                      |
| --------- | -------------------------- |
| SRP       | One reason to change       |
| ISP       | Don’t force unused methods |

**Design Smell Checklist**

If you see:
- Large interfaces
- Many unused methods
- Fake implementations

👉 Apply ISP immediately.